In [17]:

install.packages("DBI", repos="https://cloud.r-project.org")
install.packages("RSQLite", repos="https://cloud.r-project.org")
library(DBI)
library(RSQLite)
con <- dbConnect(SQLite(), "askisi1_bd.sqlite")
dbExecute(con, "DROP TABLE IF EXISTS DILWNEI;")
dbExecute(con, "DROP TABLE IF EXISTS PROTEINEI;")
dbExecute(con, "DROP TABLE IF EXISTS FOITHTHS;")
dbExecute(con, "DROP TABLE IF EXISTS MATHIMA;")
dbExecute(con, "DROP TABLE IF EXISTS TMHMA;")
dbExecute(con, "DROP TABLE IF EXISTS PANEPISTIMIO;")
dbExecute(con, "DROP TABLE IF EXISTS SUGGRAMMA;")
dbExecute(con, "
CREATE TABLE PANEPISTIMIO (
    KodikosP INTEGER PRIMARY KEY,
    Onomasia TEXT);
")
dbExecute(con, "
CREATE TABLE TMHMA (
    KodikosT INTEGER PRIMARY KEY,
    Onomasia TEXT,
    KodikosP INTEGER,
    FOREIGN KEY (KodikosP) REFERENCES PANEPISTIMIO(KodikosP));
")
dbExecute(con, "
CREATE TABLE MATHIMA (
    KodikosM INTEGER PRIMARY KEY,
    Titlos TEXT,
    EtosSpoudwn INTEGER,
    ExaminoSpoudwn INTEGER,
    KodikosT INTEGER,
    FOREIGN KEY (KodikosT) REFERENCES TMHMA(KodikosT));
")
dbExecute(con, "
CREATE TABLE FOITHTHS (
    KodikosF INTEGER PRIMARY KEY,
    EtosEggrafis INTEGER,
    KodikosT INTEGER,
    FOREIGN KEY (KodikosT) REFERENCES TMHMA(KodikosT));
")
dbExecute(con, "
CREATE TABLE SUGGRAMMA (
    KodikosS INTEGER PRIMARY KEY,
    Titlos TEXT,
    Suggrafeis TEXT,
    ArithmosEkdosis INTEGER,
    EtosEkdosis INTEGER,
    EkdotikosOikos TEXT,
    Timi REAL,
    URL TEXT);
")
dbExecute(con, "
CREATE TABLE PROTEINEI (
    KodikosM INTEGER,
    KodikosS INTEGER,
    Seira INTEGER,
    PRIMARY KEY (KodikosM, KodikosS),
    FOREIGN KEY (KodikosM) REFERENCES MATHIMA(KodikosM),
    FOREIGN KEY (KodikosS) REFERENCES SUGGRAMMA(KodikosS));
")
dbExecute(con, "
CREATE TABLE DILWNEI (
    KodikosF INTEGER,
    KodikosM INTEGER,
    KodikosS INTEGER,
    PRIMARY KEY (KodikosF, KodikosM, KodikosS),
    FOREIGN KEY (KodikosF) REFERENCES FOITHTHS(KodikosF),
    FOREIGN KEY (KodikosM) REFERENCES MATHIMA(KodikosM),
    FOREIGN KEY (KodikosS) REFERENCES SUGGRAMMA(KodikosS));
")
#ΕΙΣΑΓΩΓΗ ΤΥΧΑΙΩΝ ΔΕΔΟΜΕΝΩΝ#
dbExecute(con, "
INSERT INTO PANEPISTIMIO VALUES
(1, 'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ'),
(2, 'ΑΠΘ');
")
dbExecute(con, "
INSERT INTO TMHMA VALUES
(10, 'ΜΑΘΗΜΑΤΙΚΟ', 1),
(20, 'ΠΛΗΡΟΦΟΡΙΚΗ', 1),
(30, 'ΦΥΣΙΚΟ', 2);
")
dbExecute(con, "
INSERT INTO MATHIMA VALUES
(100, 'ΒΑΣΕΙΣ ΔΕΔΟΜΕΝΩΝ', 2, 4, 20),
(101, 'ΑΛΓΕΒΡΑ', 1, 1, 10),
(102, 'ΑΝΑΛΥΣΗ', 1, 2, 10);
")
dbExecute(con, "
INSERT INTO FOITHTHS VALUES
(1000, 2022, 10),
(1001, 2021, 10),
(1002, 2023, 20),
(1003, 2022, 30);
")
dbExecute(con, "
INSERT INTO SUGGRAMMA VALUES
(500, 'Εισαγωγή στις ΒΔ', 'Α. Συγγραφέας', 1, 2018, 'ΚΛΕΙΔΑΡΙΘΜΟΣ', 35.5, 'url1'),
(501, 'Άλγεβρα Ι', 'Β. Συγγραφέας', 2, 2015, 'ΠΑΠΑΣΩΤΗΡΙΟΥ', NULL, 'url2'),
(502, 'Ανάλυση Ι', 'Γ. Συγγραφέας', 1, 2010, 'GUTENBERG', 28.0, 'url3'),
(503, 'Προχωρημένες ΒΔ', 'Δ. Συγγραφέας', 1, 2020, 'ΚΛΕΙΔΑΡΙΘΜΟΣ', 42.0, 'url4');
")
dbExecute(con, "
INSERT INTO PROTEINEI VALUES
(100, 500, 1),
(100, 503, 2),
(101, 501, 1),
(102, 502, 1);
")
dbExecute(con, "
INSERT INTO DILWNEI VALUES
(1000, 101, 501),
(1000, 102, 502),
(1001, 101, 501),
(1002, 100, 500),
(1002, 100, 503);
")
# ΕΝΔΕΙΚΤΙΚΕΣ ΑΠΑΝΤΗΣΕΙΣ
cat("\n--- ΕΡΩΤΗΜΑ 1 ---\n")
print(dbGetQuery(con, "
SELECT KodikosS, Titlos, Suggrafeis, ArithmosEkdosis, EtosEkdosis, EkdotikosOikos
FROM SUGGRAMMA
WHERE Timi IS NULL;
"))
cat("\n--- ΕΡΩΤΗΜΑ 2 ---\n")
print(dbGetQuery(con, "
SELECT f.KodikosF, f.EtosEggrafis
FROM FOITHTHS f
JOIN TMHMA t ON f.KodikosT = t.KodikosT
JOIN PANEPISTIMIO p ON t.KodikosP = p.KodikosP
WHERE t.Onomasia = 'ΜΑΘΗΜΑΤΙΚΟ'
AND p.Onomasia = 'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ';
"))
cat("\n--- ΕΡΩΤΗΜΑ 3 ---\n")
print(dbGetQuery(con, "
SELECT COUNT(DISTINCT KodikosF) AS Plithos
FROM DILWNEI;
"))
cat("\n--- ΕΡΩΤΗΜΑ 4 ---\n")
print(dbGetQuery(con, "
SELECT s.EkdotikosOikos, COUNT(*) AS Plithos
FROM DILWNEI d
JOIN SUGGRAMMA s ON d.KodikosS = s.KodikosS
GROUP BY s.EkdotikosOikos
ORDER BY s.EkdotikosOikos;
"))
cat("\n--- ΕΡΩΤΗΜΑ 5 ---\n")
print(dbGetQuery(con, "
SELECT EkdotikosOikos,
MIN(Timi) AS Elaxisti,
MAX(Timi) AS Megisti
FROM SUGGRAMMA
GROUP BY EkdotikosOikos
HAVING COUNT(*) > 1;
"))
cat("\n--- ΕΡΩΤΗΜΑ 6 ---\n")
print(dbGetQuery(con, "
SELECT KodikosS, COUNT(DISTINCT KodikosF) AS Plithos
FROM DILWNEI
GROUP BY KodikosS
HAVING COUNT(DISTINCT KodikosF) > 2
ORDER BY Plithos DESC;
"))
cat("\n--- ΕΡΩΤΗΜΑ 7 ---\n")
print(dbGetQuery(con, "
SELECT Titlos, Suggrafeis, ArithmosEkdosis, EtosEkdosis, EkdotikosOikos
FROM SUGGRAMMA
WHERE EtosEkdosis = (SELECT MIN(EtosEkdosis) FROM SUGGRAMMA);
"))
cat("\n--- ΕΡΩΤΗΜΑ 8 ---\n")
print(dbGetQuery(con, "
SELECT DISTINCT d.KodikosF, m.KodikosT, m.KodikosM, m.Titlos
FROM DILWNEI d
JOIN MATHIMA m ON d.KodikosM = m.KodikosM
WHERE d.KodikosM IN (
    SELECT KodikosM
    FROM PROTEINEI
    GROUP BY KodikosM
    HAVING COUNT(*) = 1
)
ORDER BY m.KodikosT;
"))
cat("\n--- ΕΡΩΤΗΜΑ 9 ---\n")
print(dbGetQuery(con, "
SELECT DISTINCT s.EkdotikosOikos
FROM SUGGRAMMA s
WHERE s.EkdotikosOikos NOT IN (
    SELECT s2.EkdotikosOikos
    FROM PROTEINEI p
    JOIN SUGGRAMMA s2 ON p.KodikosS = s2.KodikosS
    JOIN MATHIMA m ON p.KodikosM = m.KodikosM
    JOIN TMHMA t ON m.KodikosT = t.KodikosT
    JOIN PANEPISTIMIO pa ON t.KodikosP = pa.KodikosP
    WHERE pa.Onomasia = 'ΠΑΝΕΠΙΣΤΗΜΙΟ ΠΑΤΡΩΝ'
)
ORDER BY EkdotikosOikos;
"))
cat("\n--- ΕΡΩΤΗΜΑ 10 ---\n")
print(dbGetQuery(con, "
SELECT KodikosF, COUNT(*) AS Plithos
FROM DILWNEI
GROUP BY KodikosF
HAVING COUNT(*) = (
    SELECT MAX(cnt)
    FROM (
        SELECT COUNT(*) AS cnt
        FROM DILWNEI
        GROUP BY KodikosF
    )
);
"))

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



[1] 0

[1] 0

[1] 0

[1] 0

...

[1] 4

[1] 4

[1] 4

[1] 5


--- ΕΡΩΤΗΜΑ 1 ---
  KodikosS    Titlos    Suggrafeis ArithmosEkdosis EtosEkdosis EkdotikosOikos
1      501 Άλγεβρα Ι Β. Συγγραφέας               2        2015   ΠΑΠΑΣΩΤΗΡΙΟΥ

--- ΕΡΩΤΗΜΑ 2 ---
  KodikosF EtosEggrafis
1     1000         2022
2     1001         2021

--- ΕΡΩΤΗΜΑ 3 ---
  Plithos
1       3

--- ΕΡΩΤΗΜΑ 4 ---
  EkdotikosOikos Plithos
1      GUTENBERG       1
2   ΚΛΕΙΔΑΡΙΘΜΟΣ       2
3   ΠΑΠΑΣΩΤΗΡΙΟΥ       2

--- ΕΡΩΤΗΜΑ 5 ---
  EkdotikosOikos Elaxisti Megisti
1   ΚΛΕΙΔΑΡΙΘΜΟΣ     35.5      42

--- ΕΡΩΤΗΜΑ 6 ---
[1] KodikosS Plithos 
<0 rows> (or 0-length row.names)

--- ΕΡΩΤΗΜΑ 7 ---
     Titlos    Suggrafeis ArithmosEkdosis EtosEkdosis EkdotikosOikos
1 Ανάλυση Ι Γ. Συγγραφέας               1        2010      GUTENBERG

--- ΕΡΩΤΗΜΑ 8 ---
  KodikosF KodikosT KodikosM  Titlos
1     1000       10      101 ΑΛΓΕΒΡΑ
2     1000       10      102 ΑΝΑΛΥΣΗ
3     1001       10      101 ΑΛΓΕΒΡΑ

--- ΕΡΩΤΗΜΑ 9 ---
[1] EkdotikosOikos
<0 rows> (or 0-length row.names)

--- ΕΡΩΤΗΜΑ 10 ---
